Import

In [51]:
import numpy as np
import pandas as pd


Data - SLP

In [71]:
# Standard-Lastprofil für 2019 (stündlich)
file_path = r"D:\Dokumente\Studium\20242025 WS\WattWise\Repo\wattWise\data\slp\GGV_SLP_1000_MWh_2019_01.xlsx"
sheet_name = "15Min - SLP 2019" # Single Load Profile 2019

df_standard_load_profile_2019 = pd.read_excel(file_path, sheet_name=sheet_name)
df_standard_load_profile_2019



,Messwert-Nr.,Wochentag,Datum,Zeitstempel von,Zeitstempel bis,MP von,MP bis,H00 [kW],H00 [kWh]
0,1,2,2019-01-01,2019-01-01 00:00:00.000,2019-01-01 00:15:00.000,00:00,00:15,145.200,36.300
1,2,2,2019-01-01,2019-01-01 00:15:00.000,2019-01-01 00:30:00.000,00:15,00:30,140.112,35.028
2,3,2,2019-01-01,2019-01-01 00:30:00.000,2019-01-01 00:45:00.000,00:30,00:45,135.720,33.930
3,4,2,2019-01-01,2019-01-01 00:45:00.000,2019-01-01 01:00:00.000,00:45,01:00,130.732,32.683
4,5,2,2019-01-01,2019-01-01 01:00:00.000,2019-01-01 01:15:00.000,01:00,01:15,127.236,31.809
...,...,...,...,...,...,...,...,...,...
35035,35036,2,2019-12-31,2019-12-31 22:44:59.999,2019-12-31 22:59:59.999,22:45,23:00,158.172,39.543
35036,35037,2,2019-12-31,2019-12-31 22:59:59.999,2019-12-31 23:14:59.999,23:00,23:15,158.672,39.668
35037,35038,2,2019-12-31,2019-12-31 23:14:59.999,2019-12-31 23:29:59.999,23:15,23:30,155.480,38.870
35038,35039,2,2019-12-31,2019-12-31 23:29:59.999,2019-12-31 23:44:59.999,23:30,23:45,151.888,37.972


In [72]:
def fill_missing_hours(df: pd.DataFrame, time_col: str = "Time", value_cols: list = None,
                       year: int = 2019, fill_value=None) -> pd.DataFrame:
    """
    Erkennt fehlende Stunden in einem Zeitreihen-DataFrame und füllt sie optional mit fill_value (z. B. 0 oder NaN).

    Args:
        df (DataFrame): Zeitreihen-DataFrame mit einer Zeitspalte.
        time_col (str): Name der Spalte mit Timestamps.
        value_cols (list): Spalten, die beibehalten/gefüllt werden sollen. Wenn None → alle außer time_col.
        year (int): Jahr, für das die Stunden geprüft werden sollen.
        fill_value: Wert zum Auffüllen (z. B. 0, np.nan oder dict).

    Returns:
        DataFrame mit allen 8760 Stunden (oder 8784 bei Schaltjahr), vollständig aufgefüllt.
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df.set_index(time_col, inplace=True)

    # Generiere kompletten Stundenindex für das Jahr
    start = f"{year}-01-01 00:00"
    end = f"{year}-12-31 23:00"
    full_index = pd.date_range(start=start, end=end, freq="H")

    # Bestimme Spalten
    if value_cols is None:
        value_cols = [col for col in df.columns if col != time_col]

    # Reindex & auffüllen
    df_full = df.reindex(full_index)

    if fill_value is not None:
        df_full[value_cols] = df_full[value_cols].fillna(fill_value)

    df_full.reset_index(inplace=True)
    df_full.rename(columns={"index": time_col}, inplace=True)

    return df_full

In [73]:
#Prepare DataFrame
df_standard_load_profile_2019.drop(columns=['Messwert-Nr.','MP von', 'MP bis'], inplace=True)

#Fasse Zeitstempel pro Stunde zusammen und summiere H00 [kW] in neuer df
df_standard_load_profile_2019['Time'] = df_standard_load_profile_2019['Zeitstempel von'].dt.floor('h')

df_agg = df_standard_load_profile_2019.groupby('Time').agg({'H00 [kW]': 'sum'}).reset_index()
df_agg['H00 [kWh]'] = df_agg['H00 [kW]'] / 4 

df_standard_load_profile_2019 = fill_missing_hours(df=df_agg, time_col='Time', value_cols=['H00 [kWh]'],
    year=2019,
    fill_value=0 # oder np.nan
)

df_standard_load_profile_2019

C:\Users\Hannah Heimpel\AppData\Local\Temp\ipykernel_1060\386875733.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=start, end=end, freq="H")


,Time,H00 [kW],H00 [kWh]
0,2019-01-01 00:00:00,551.764,137.941
1,2019-01-01 01:00:00,492.584,123.146
2,2019-01-01 02:00:00,448.376,112.094
3,2019-01-01 03:00:00,418.236,104.559
4,2019-01-01 04:00:00,386.208,96.552
...,...,...,...
8755,2019-12-31 19:00:00,656.444,164.111
8756,2019-12-31 20:00:00,574.216,143.554
8757,2019-12-31 21:00:00,557.048,139.262
8758,2019-12-31 22:00:00,640.876,160.219


In [74]:
#add 'Wochentag'
df_standard_load_profile_2019['Weekday'] = df_standard_load_profile_2019['Time'].dt.day_name()

#add Arbeitstag oder Wochenende als neue Spalte
df_standard_load_profile_2019['Weekend'] = df_standard_load_profile_2019['Weekday'].apply(lambda x: 1 if x in ['Saturday', 'Sunday'] else 0)

#add Hour and save as int
df_standard_load_profile_2019['Hour'] = df_standard_load_profile_2019['Time'].dt.time
df_standard_load_profile_2019['Hour'] = df_standard_load_profile_2019['Hour'].apply(lambda x: x.hour)

#Normalizing the values to a "average household" with 3000 kWh  
df_standard_load_profile_2019['Normalized Load'] = df_standard_load_profile_2019['H00 [kWh]'] /df_standard_load_profile_2019['H00 [kWh]'].sum()*3000

#Save an seaonality index for each hour on a annual mean of 1.0
df_standard_load_profile_2019['Seasonal Factor'] = df_standard_load_profile_2019['Normalized Load'] / df_standard_load_profile_2019['Normalized Load'].mean()

Data - 2 week customer data

_1. Input_

In [75]:
# 2 Wochen Kundenverbrauch für 2020 (stündlich) -- hier ggf. Input ändern #TODO
df_random_two_week_customer_load_2020 = pd.DataFrame(np.random.rand(24 * 14) * 2, columns=['Customer_Load'])

#add random dates for 2 weeks in 2020 on hourly basis -- hier gfg. Input ändern #TODO
start_date = pd.to_datetime('2020-01-01 00:00:00')
end_date = pd.to_datetime('2020-01-14 23:00:00')
# Create a date range with hourly frequency
date_range = pd.date_range(start=start_date, end=end_date, freq='H')
df_random_two_week_customer_load_2020['Time'] = date_range

# Jahresverbrauch des Kunden in kWh
customer_annual_consumption_preyear = 3000 # kWh #TODO hier ggf. Input ändern

df_random_two_week_customer_load_2020


C:\Users\Hannah Heimpel\AppData\Local\Temp\ipykernel_1060\2627068670.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start=start_date, end=end_date, freq='H')


,Customer_Load,Time
0,0.749080,2020-01-01 00:00:00
1,1.901429,2020-01-01 01:00:00
2,1.463988,2020-01-01 02:00:00
3,1.197317,2020-01-01 03:00:00
4,0.312037,2020-01-01 04:00:00
...,...,...
331,1.515692,2020-01-14 19:00:00
332,0.028787,2020-01-14 20:00:00
333,0.232145,2020-01-14 21:00:00
334,0.092005,2020-01-14 22:00:00


In [76]:
def expand_load_profile_to_full_year(df_input: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Wiederholt ein stündliches 2-Wochen-Lastprofil über ein ganzes Jahr (keine Saisonalität).

    Args:
        df_input (pd.DataFrame): DataFrame mit 'Time' und 'Customer Load' with two week data
        year (int): Jahr, auf das expandiert werden soll.

    Returns:
        pd.DataFrame: DataFrame mit stündlichem Verbrauch für das ganze Jahr.
    """
    df_input = df_input.copy()
    df_input['Time'] = pd.to_datetime(df_input['Time'])
    df_input = df_input.set_index('Time')

    hours_per_year = 8784 if pd.Timestamp(f"{year}-12-31 23:00").is_leap_year else 8760
    repeated_times = hours_per_year // len(df_input) + 1
    df_full = pd.concat([df_input] * repeated_times, ignore_index=False)
    df_full = df_full.iloc[:hours_per_year]

    df_full.index = pd.date_range(start=f"{year}-01-01 00:00", periods=hours_per_year, freq="H")

    
    return df_full.reset_index().rename(columns={"index": "Time"})


df_full_year = expand_load_profile_to_full_year(df_random_two_week_customer_load_2020, 2020)
df_full_year

print("Summe vor Skalierung:", df_full_year['Customer_Load'].sum())

Summe vor Skalierung: 8624.020881471504


C:\Users\Hannah Heimpel\AppData\Local\Temp\ipykernel_1060\262372465.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_full.index = pd.date_range(start=f"{year}-01-01 00:00", periods=hours_per_year, freq="H")


_2. Skalierung des Kundenprofils_ auf Jahreskundeneverbrauch

In [77]:

# Skalierungsfaktor berechnen, um auf den Jahresverbrauch zu normieren.
def scale_annual_consumption(df_full_year: pd.DataFrame, customer_annual_consumption_preyear: float) -> pd.DataFrame:
    """
    Skaliert den Jahresverbrauch eines synthetisch erweiterten Lastprofils auf den bekannten Jahresverbrauch.

    Args:
        df_full_year (pd.DataFrame): DataFrame mit Spalten 'Time' und 'Customer Load' (in kWh pro Stunde) für ein gesamtes Jahr
        customer_annual_consumption_preyear (float): Ziel-Jahresverbrauch in kWh.

    Returns:
        pd.DataFrame: DataFrame mit skaliertem 'Customer_Load'.
    """
    df_full_year_scaled = df_full_year.copy()

    total_consumption_current = df_full_year_scaled['Customer_Load'].sum()
    if total_consumption_current == 0:
        raise ValueError("Gesamtverbrauch ist 0 – Skalierung nicht möglich.")

    scaling_factor = customer_annual_consumption_preyear / total_consumption_current

    df_full_year_scaled['Customer_Load'] = df_full_year_scaled['Customer_Load'] * scaling_factor

    return df_full_year_scaled


df_full_year_scaled = scale_annual_consumption(df_full_year, customer_annual_consumption_preyear)
print("Summe nach Skalierung:", df_full_year_scaled['Customer_Load'].sum())

df_full_year_scaled




Summe nach Skalierung: 3000.0


,Time,Customer_Load
0,2020-01-01 00:00:00,0.260579
1,2020-01-01 01:00:00,0.661442
2,2020-01-01 02:00:00,0.509271
3,2020-01-01 03:00:00,0.416505
4,2020-01-01 04:00:00,0.108547
...,...,...
8779,2020-12-31 19:00:00,0.632643
8780,2020-12-31 20:00:00,0.180041
8781,2020-12-31 21:00:00,0.460937
8782,2020-12-31 22:00:00,0.216867


Monte Carlo

In [79]:
def simulate_stochastic_seasonality_montecarlo(df_load: pd.DataFrame,
                                                slp_profile: pd.DataFrame,
                                                slp_col: str = "Seasonality Factor",
                                                n_simulations: int = 1000,
                                                std_dev: float = 0.1,
                                                seed: int = None) -> pd.DataFrame:
    """
    Erzeugt mehrere stochastisch modifizierte Lastprofile basierend auf SLP und MC-Variation.

    Args:
        df_load (DataFrame): flaches Lastprofil, Spalte 'Customer_Load', über das ganze Jahr
        slp_profile (DataFrame): saisonales Standardlastprofil, Spalte 'Seasonality Factor'
        
        n_simulations (int): Anzahl der MC-Simulationen
        std_dev (float): Variation der Saisonalität (z. B. 0.1 = ±10 %)
        seed (int): Seed für Reproduzierbarkeit

    Returns:
        DataFrame mit Spalten ['Time', sim_1, sim_2, ..., sim_n], jeweils eine simulierte Verbrauchskurve.
    """

    if seed is not None:
        np.random.seed(seed)

#Wenn df_load Schaltjahr, dann lösche den 29. Februar
    if len(df_load) == 8784:
        df_load = df_load[~((df_load['Time'].dt.month == 2) & (df_load['Time'].dt.day == 29))].reset_index(drop=True)

    print("DFLoad:", len(df_load))
    print("SLP:", len(slp_profile))

    if len(df_load) != len(slp_profile):
        raise ValueError("df_load und slp_profile müssen gleich viele Zeilen haben")
    
    base_load = df_load['Customer_Load'].values
    seasonality_raw = slp_profile[slp_col].values
    hours = len(base_load)

    # Matrix mit Normalverteiltem Noise um 1
    noise_matrix = np.random.normal(loc=1.0, scale=std_dev, size=(hours, n_simulations))

    # SLP * Rauschfaktor → neue saisonale Faktoren
    seasonality_matrix = seasonality_raw[:, np.newaxis] * noise_matrix

    # Endgültige Last: base_load * stochastische SLP-Faktoren
    load_matrix = base_load[:, np.newaxis] * seasonality_matrix

    # DataFrame mit Time Index und Simulationsspalten
    time_index = df_load['Time'] if 'Time' in df_load.columns else pd.date_range("2020-01-01", periods=hours, freq="H")
    df_simulations = pd.DataFrame(load_matrix, index=time_index, columns=[f"sim_{i+1}" for i in range(n_simulations)])
    return df_simulations

df_simulated = simulate_stochastic_seasonality_montecarlo(
    df_load=df_full_year_scaled,         # mein flaches Profil
    slp_profile=df_standard_load_profile_2019,                  # z. B. SLP auf 3000 kWh
    slp_col="Normalized Load",
    n_simulations=100,
    std_dev=0.08,
    seed=42
)

DFLoad: 8760
SLP: 8760


In [81]:
df_simulated.head()

,sim_1,sim_2,sim_3,sim_4,sim_5,sim_6,sim_7,sim_8,sim_9,sim_10,...,sim_91,sim_92,sim_93,sim_94,sim_95,sim_96,sim_97,sim_98,sim_99,sim_100
Time,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,0.112119,0.106641,0.113421,0.120972,0.105814,0.105814,0.121457,0.114454,0.103784,0.112514,...,0.108671,0.116190,0.101777,0.105007,0.104451,0.095208,0.110388,0.110086,0.107878,0.105810
2020-01-01 01:00:00,0.216693,0.236138,0.237662,0.228678,0.241209,0.252260,0.281235,0.247774,0.249396,0.242906,...,0.235633,0.261103,0.248547,0.220009,0.247747,0.251894,0.227083,0.247367,0.245500,0.222018
2020-01-01 02:00:00,0.176161,0.178942,0.186097,0.185696,0.152384,0.158410,0.178315,0.178298,0.178315,0.224044,...,0.168407,0.164504,0.163184,0.182899,0.176150,0.161765,0.183584,0.175469,0.182395,0.179885
2020-01-01 03:00:00,0.121984,0.124793,0.138459,0.137028,0.130430,0.131874,0.144002,0.124465,0.136366,0.128535,...,0.121258,0.135790,0.116849,0.149790,0.142975,0.125744,0.112743,0.144799,0.129451,0.143586
2020-01-01 04:00:00,0.027431,0.029934,0.031454,0.031559,0.030309,0.033008,0.028756,0.031083,0.031744,0.032735,...,0.031421,0.035164,0.031636,0.029275,0.035272,0.032797,0.028832,0.030963,0.029239,0.027963
